In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# 1. PATHS
# ============================================================

#ADMISSIONS_PATH = r"C:\path\to\mimiciv\hosp\admissions.csv.gz"
#DIAGNOSES_PATH = r"C:\path\to\mimiciv\hosp\diagnoses_icd.csv.gz"

ADMISSIONS_PATH = r"C:\Users\oluwa\OneDrive\Desktop\CoC Files\Current Literature\DNTransformerLLMsMIMIC\outputs-20260805T024105Z-1-001\outputs\mimic-iv-3.1\mimic-iv-3.1\hosp\admissions.csv.gz"
DIAGNOSES_PATH = r"C:\Users\oluwa\OneDrive\Desktop\CoC Files\Current Literature\DNTransformerLLMsMIMIC\outputs-20260805T024105Z-1-001\outputs\mimic-iv-3.1\mimic-iv-3.1\hosp\diagnoses_icd.csv.gz"

OUTPUT_DIR = Path(
    r"C:\Users\oluwa\OneDrive\Desktop\CoC Files\Current Literature\DNTransformerLLMsMIMIC\outputs-20260805T024105Z-1-001\outputs"
)

# ============================================================
# 2. LOAD MIMIC FILES
# ============================================================

admissions = pd.read_csv(
    ADMISSIONS_PATH,
    usecols=[
        "subject_id",
        "hadm_id",
        "admittime",
        "dischtime"
    ]
)

diagnoses = pd.read_csv(
    DIAGNOSES_PATH,
    usecols=[
        "subject_id",
        "hadm_id",
        "seq_num",
        "icd_code",
        "icd_version"
    ]
)

admissions["admittime"] = pd.to_datetime(admissions["admittime"])
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"])

# Clean ICD codes
diagnoses["icd_code"] = (
    diagnoses["icd_code"]
    .astype(str)
    .str.upper()
    .str.replace(".", "", regex=False)
)

# ============================================================
# 3. IDENTIFY DIABETES
# ============================================================

def is_diabetes(row):

    code = row["icd_code"]
    version = row["icd_version"]

    # ICD-9 diabetes
    if version == 9:
        return code.startswith("250")

    # ICD-10 diabetes families
    if version == 10:
        return code.startswith(
            ("E08", "E09", "E10", "E11", "E13")
        )

    return False


diagnoses["diabetes_code"] = diagnoses.apply(
    is_diabetes,
    axis=1
)

# ============================================================
# 4. IDENTIFY DIABETIC NEPHROPATHY / DIABETIC CKD
# ============================================================

def is_dn(row):

    code = row["icd_code"]
    version = row["icd_version"]

    # ICD-9: diabetes with renal manifestations
    if version == 9:
        return code.startswith("2504")

    # ICD-10:
    # .21 = diabetic nephropathy
    # .22 = diabetes with diabetic CKD
    if version == 10:

        dn_codes = (
            "E0821", "E0822",
            "E0921", "E0922",
            "E1021", "E1022",
            "E1121", "E1122",
            "E1321", "E1322"
        )

        return code.startswith(dn_codes)

    return False


diagnoses["dn_code"] = diagnoses.apply(
    is_dn,
    axis=1
)

# ============================================================
# 5. CREATE ADMISSION-LEVEL FLAGS
# ============================================================

admission_flags = (
    diagnoses
    .groupby(
        ["subject_id", "hadm_id"],
        as_index=False
    )
    .agg(
        diabetes_flag=("diabetes_code", "max"),
        dn_flag=("dn_code", "max")
    )
)

adm = admissions.merge(
    admission_flags,
    on=["subject_id", "hadm_id"],
    how="left"
)

adm["diabetes_flag"] = (
    adm["diabetes_flag"]
    .fillna(False)
    .astype(int)
)

adm["dn_flag"] = (
    adm["dn_flag"]
    .fillna(False)
    .astype(int)
)

# ============================================================
# 6. KEEP PATIENTS WITH DIABETES
# ============================================================

diabetes_patients = adm.loc[
    adm["diabetes_flag"] == 1,
    "subject_id"
].unique()

adm = adm[
    adm["subject_id"].isin(diabetes_patients)
].copy()

# Order every patient's admissions
adm = adm.sort_values(
    ["subject_id", "admittime", "hadm_id"]
)

adm["visit_number"] = (
    adm.groupby("subject_id")
    .cumcount()
    + 1
)

# ============================================================
# 7. CREATE LONGITUDINAL INDEX ADMISSION
# ============================================================

cohort_rows = []

for subject_id, patient in adm.groupby("subject_id"):

    patient = patient.sort_values("admittime").copy()

    # Need at least TWO admissions
    if len(patient) < 2:
        continue

    dn_visits = patient[
        patient["dn_flag"] == 1
    ]

    # --------------------------------------------------------
    # DN POSITIVE
    # First admission where DN appears
    # --------------------------------------------------------

    if len(dn_visits) > 0:

        index_row = dn_visits.iloc[0]

        earlier_visits = patient[
            patient["admittime"]
            < index_row["admittime"]
        ]

        # Must have prior history
        if len(earlier_visits) == 0:
            continue

        label = 1

    # --------------------------------------------------------
    # DN NEGATIVE
    # Never diagnosed with DN
    # Use last admission as index
    # --------------------------------------------------------

    else:

        index_row = patient.iloc[-1]

        earlier_visits = patient.iloc[:-1]

        if len(earlier_visits) == 0:
            continue

        label = 0

    cohort_rows.append(
        {
            "subject_id": subject_id,

            "index_hadm_id":
                int(index_row["hadm_id"]),

            "index_admittime":
                index_row["admittime"],

            "primary_icd_dn_label":
                label,

            "n_prior_visits":
                len(earlier_visits),

            "index_visit_number":
                int(index_row["visit_number"])
        }
    )

# ============================================================
# 8. CREATE COHORT
# ============================================================

longitudinal_labels = pd.DataFrame(
    cohort_rows
)

print(
    "Patients:",
    len(longitudinal_labels)
)

print("\nDN distribution:")

print(
    longitudinal_labels[
        "primary_icd_dn_label"
    ].value_counts()
)

print("\nDN rate:")

print(
    longitudinal_labels[
        "primary_icd_dn_label"
    ].mean()
)

print("\nPrior visit distribution:")

print(
    longitudinal_labels[
        "n_prior_visits"
    ].describe()
)

# ============================================================
# 9. SAVE
# ============================================================

output_file = (
    OUTPUT_DIR /
    "cohort_labels_longitudinal.parquet"
)

longitudinal_labels.to_parquet(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)

C:\Users\oluwa\AppData\Local\Temp\ipykernel_5200\1619179450.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
C:\Users\oluwa\AppData\Local\Temp\ipykernel_5200\1619179450.py:147: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


Patients: 24358

DN distribution:
primary_icd_dn_label
0    19077
1     5281
Name: count, dtype: int64

DN rate:
0.21680761967320797

Prior visit distribution:
count    24358.000000
mean         3.724649
std          5.453654
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max        237.000000
Name: n_prior_visits, dtype: float64

Saved:
C:\Users\oluwa\OneDrive\Desktop\CoC Files\Current Literature\DNTransformerLLMsMIMIC\outputs-20260805T024105Z-1-001\outputs\cohort_labels_longitudinal.parquet
